# 两域颜色校准：冻结主干和旧融合
与上一轮使用相同三个Input。先GPU缓存50张预测，再CPU拟合正则化RGB校准器；不重新训练ESDNet。保持GPU和Internet启用，直接Run All。

In [ ]:
%pip install -q gdown pyyaml scikit-image


In [ ]:
import types
J=types.ModuleType("joint_helpers")
exec('import types\nH = types.ModuleType(\'baseline_helpers\')\nexec(\'"""复制整个文件到 Kaggle 新单元格执行。先安装 gdown pyyaml scikit-image。\\n独立入口：不依赖原 UHDM 数据集、不加载融合模型、不训练。\\n"""\\nfrom pathlib import Path\\nimport csv, io, json, hashlib, zipfile, sys, subprocess, importlib.util, time\\nfrom datetime import datetime\\nimport numpy as np\\nimport pandas as pd\\nfrom PIL import Image, ImageOps, ImageDraw\\nimport torch\\nimport torch.nn.functional as F\\nimport yaml\\nfrom skimage.metrics import structural_similarity\\nfrom tqdm.auto import tqdm\\nfrom IPython.display import display, FileLink\\nDATA_SOURCE = None\\nLOCAL_WEIGHT = None\\nREGION = 768\\nSCALES = (1.0, 0.5)\\nMAX_PAIRS = 20\\nROOT = Path(\\\'/kaggle/working/dcid_baseline\\\')\\n\\ndef sha_file(path):\\n    h = hashlib.sha256()\\n    with Path(path).open(\\\'rb\\\') as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\'\\\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\nclass PairSource:\\n\\n    def __init__(self, source):\\n        self.path = Path(source)\\n        self.archive = zipfile.ZipFile(self.path) if self.path.suffix.lower() == \\\'.zip\\\' else None\\n        if self.archive:\\n            members = [n for n in self.archive.namelist() if n.endswith(\\\'_manifest_draft.csv\\\')]\\n            if len(members) != 1:\\n                raise RuntimeError(f\\\'ZIP 内应有唯一原始清单，实际为 {members}\\\')\\n            self.manifest_bytes = self.archive.read(members[0])\\n            self.prefix = members[0].rsplit(\\\'/\\\', 1)[0] + \\\'/\\\' if \\\'/\\\' in members[0] else \\\'\\\'\\n        else:\\n            self.manifest_bytes = self.path.read_bytes()\\n        self.rows = list(csv.DictReader(io.StringIO(self.manifest_bytes.decode(\\\'utf-8-sig\\\'), newline=\\\'\\\')))\\n        if not self.rows:\\n            raise RuntimeError(\\\'清单为空\\\')\\n        if not {\\\'sample_id\\\', \\\'input\\\', \\\'target\\\'} <= set(self.rows[0]):\\n            raise RuntimeError(\\\'清单缺少必要字段\\\')\\n        ids = [r[\\\'sample_id\\\'] for r in self.rows]\\n        if len(ids) != len(set(ids)):\\n            raise RuntimeError(\\\'sample_id 重复，请检查清单\\\')\\n\\n    def read_bytes(self, row, key):\\n        name = row[key].replace(\\\'\\\\\\\\\\\', \\\'/\\\')\\n        relative = \\\'images/\\\' + name.split(\\\'/images/\\\', 1)[1] if \\\'/images/\\\' in name else name\\n        if self.archive:\\n            return self.archive.read(self.prefix + relative)\\n        p = Path(name)\\n        if not p.is_file():\\n            p = self.path.parent / relative\\n        if not p.is_file():\\n            raise FileNotFoundError(f\\\'{key}: {name}，也未找到 {p}\\\')\\n        return p.read_bytes()\\n\\n    def read(self, row, key):\\n        with Image.open(io.BytesIO(self.read_bytes(row, key))) as image:\\n            return np.array(image.convert(\\\'RGB\\\'), copy=True)\\n\\ndef locate_source():\\n    if DATA_SOURCE is not None:\\n        return Path(DATA_SOURCE)\\n    preferred = Path(\\\'/kaggle/working/dcid_hf_subset/train_iphone_1_part1_20_2026_manifest_draft.csv\\\')\\n    if preferred.is_file():\\n        return preferred\\n    for pattern in [\\\'*_manifest_draft.csv\\\', \\\'*_with_images.zip\\\']:\\n        found = sorted((p for base in [Path(\\\'/kaggle/working\\\'), Path(\\\'/kaggle/input\\\')] if base.exists() for p in base.rglob(pattern)))\\n        if len(found) == 1:\\n            return found[0]\\n        if len(found) > 1:\\n            raise RuntimeError(f\\\'候选不唯一，请设置 DATA_SOURCE：{found}\\\')\\n    raise FileNotFoundError(\\\'没有找到 DCID 清单或 ZIP。请恢复下载结果，或将原图子集挂载为 Kaggle Input。\\\')\\n\\ndef load_official_model():\\n    repo = Path(\\\'/kaggle/working/moire_project/UHDM\\\')\\n    if not (repo / \\\'model/nets.py\\\').is_file():\\n        repo = ROOT / \\\'UHDM\\\'\\n        if not repo.exists():\\n            subprocess.check_call([\\\'git\\\', \\\'clone\\\', \\\'--depth\\\', \\\'1\\\', \\\'https://github.com/CVMI-Lab/UHDM.git\\\', str(repo)])\\n    commit = subprocess.check_output([\\\'git\\\', \\\'-C\\\', str(repo), \\\'rev-parse\\\', \\\'HEAD\\\'], text=True).strip()\\n    if LOCAL_WEIGHT is not None:\\n        weight = Path(LOCAL_WEIGHT)\\n        if not weight.is_file():\\n            raise FileNotFoundError(weight)\\n    else:\\n        known = [Path(\\\'/kaggle/working/moire_project/uhdm_checkpoint.pth\\\'), ROOT / \\\'uhdm_checkpoint.pth\\\']\\n        weight = next((p for p in known if p.is_file()), None)\\n        if weight is None and Path(\\\'/kaggle/input\\\').exists():\\n            found = list(Path(\\\'/kaggle/input\\\').rglob(\\\'uhdm_checkpoint.pth\\\'))\\n            if len(found) > 1:\\n                raise RuntimeError(f\\\'找到多个权重，请指定 LOCAL_WEIGHT：{found}\\\')\\n            if found:\\n                weight = found[0]\\n        if weight is None:\\n            import gdown\\n            weight = ROOT / \\\'uhdm_checkpoint.pth\\\'\\n            temporary = weight.with_suffix(\\\'.download\\\')\\n            gdown.download(id=\\\'1HT_ubcAYRqzFIJ46XuPhrulJk2YFBIEl\\\', output=str(temporary), quiet=False)\\n            if not temporary.is_file():\\n                raise RuntimeError(\\\'官方权重下载失败，请挂载旧权重并填写 LOCAL_WEIGHT\\\')\\n            temporary.replace(weight)\\n    spec = importlib.util.spec_from_file_location(\\\'dcid_esdnet\\\', repo / \\\'model/nets.py\\\')\\n    module = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(module)\\n    cfg = yaml.safe_load((repo / \\\'config/uhdm_config.yaml\\\').read_text())[\\\'MODEL\\\']\\n    model = module.my_model(en_feature_num=cfg[\\\'EN_FEATURE_NUM\\\'], en_inter_num=cfg[\\\'EN_INTER_NUM\\\'], de_feature_num=cfg[\\\'DE_FEATURE_NUM\\\'], de_inter_num=cfg[\\\'DE_INTER_NUM\\\'], sam_number=cfg[\\\'SAM_NUMBER\\\'])\\n    state = torch.load(weight, map_location=\\\'cpu\\\', weights_only=True)\\n    if \\\'state_dict\\\' in state:\\n        state = state[\\\'state_dict\\\']\\n    model.load_state_dict({k.removeprefix(\\\'module.\\\'): v for k, v in state.items()}, strict=True)\\n    return (model.cuda().eval().requires_grad_(False), {\\\'repo_commit\\\': commit, \\\'weight_sha256\\\': sha_file(weight), \\\'weight_path\\\': str(weight)})\\n\\n@torch.inference_mode()\\ndef predict(model, array, scale):\\n    h, w = array.shape[:2]\\n    x = torch.from_numpy(array.copy()).permute(2, 0, 1)[None].float().cuda() / 255\\n    sh, sw = (round(h * scale), round(w * scale))\\n    if (sh, sw) != (h, w):\\n        x = F.interpolate(x, size=(sh, sw), mode=\\\'bilinear\\\', align_corners=False, antialias=True)\\n    x = F.pad(x, (0, -sw % 32, 0, -sh % 32), mode=\\\'replicate\\\')\\n    y = model(x)\\n    y = y[0] if isinstance(y, (tuple, list)) else y\\n    y = y[:, :, :sh, :sw]\\n    if (sh, sw) != (h, w):\\n        y = F.interpolate(y, size=(h, w), mode=\\\'bilinear\\\', align_corners=False)\\n    return y.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()\\n\\ndef metrics(pred, gt):\\n    pred = np.clip(pred, 0, 1).astype(np.float32)\\n    gt = gt.astype(np.float32)\\n    mse = float(np.mean((pred.astype(np.float64) - gt.astype(np.float64)) ** 2))\\n    return {\\\'psnr\\\': float(-10 * np.log10(max(mse, 1e-12))), \\\'ssim\\\': float(structural_similarity(gt, pred, data_range=1.0, channel_axis=-1, gaussian_weights=True, sigma=1.5, use_sample_covariance=False))}\\n\\ndef preview(images, path):\\n    sheet = Image.new(\\\'RGB\\\', (320 * len(images), 350), \\\'white\\\')\\n    draw = ImageDraw.Draw(sheet)\\n    for i, (label, array) in enumerate(images):\\n        im = Image.fromarray(np.round(np.clip(array, 0, 1) * 255).astype(\\\'uint8\\\'))\\n        sheet.paste(ImageOps.contain(im, (315, 315)), (i * 320, 30))\\n        draw.text((i * 320 + 5, 5), label, fill=\\\'black\\\')\\n    sheet.save(path, quality=95)\\n\\ndef main():\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\\'当前没有 GPU。先保存下载结果；GPU 空闲后，在 Notebook 设置中启用 GPU 再运行。\\\')\\n    ROOT.mkdir(parents=True, exist_ok=True)\\n    source = PairSource(locate_source())\\n    rows = source.rows[:MAX_PAIRS]\\n    fingerprints = []\\n    for row in tqdm(rows, desc=\\\'检查配对\\\'):\\n        a, g = (source.read(row, \\\'input\\\'), source.read(row, \\\'target\\\'))\\n        if a.shape != g.shape or min(a.shape[:2]) < REGION:\\n            raise RuntimeError(f\\\'配对尺寸不适用：{row[\\\'sample_id\\\']} {a.shape} {g.shape}\\\')\\n        fingerprints.append({\\\'sample_id\\\': row[\\\'sample_id\\\'], **{k + \\\'_sha256\\\': hashlib.sha256(source.read_bytes(row, k)).hexdigest() for k in [\\\'input\\\', \\\'target\\\']}})\\n    out = ROOT / datetime.now().strftime(\\\'center768_%Y%m%d_%H%M%S_%f\\\')\\n    out.mkdir()\\n    model, metadata = load_official_model()\\n    protocol = {**metadata, \\\'data_source\\\': str(source.path), \\\'manifest_sha256\\\': hashlib.sha256(source.manifest_bytes).hexdigest(), \\\'fingerprints\\\': fingerprints, \\\'region\\\': REGION, \\\'crop\\\': \\\'native-resolution center; identical for all methods\\\', \\\'scales\\\': SCALES, \\\'pairs\\\': len(rows), \\\'torch\\\': torch.__version__, \\\'gpu\\\': torch.cuda.get_device_name(0), \\\'training\\\': False, \\\'status\\\': \\\'exploratory target training-pool subset; not official test or full-image score\\\', \\\'ssim\\\': \\\'RGB, gaussian sigma1.5, population covariance, data_range1\\\', \\\'timing\\\': \\\'synchronized predict call incl CPU/GPU transfer; no speedup claim; first image includes warmup overhead\\\'}\\n    (out / \\\'protocol.json\\\').write_text(json.dumps(protocol, ensure_ascii=False, indent=2), encoding=\\\'utf-8\\\')\\n    pd.DataFrame(rows).to_csv(out / \\\'evaluated_manifest.csv\\\', index=False)\\n    results = []\\n    for index, row in enumerate(tqdm(rows, desc=\\\'DCID 基线推理\\\')):\\n        a, g = (source.read(row, \\\'input\\\'), source.read(row, \\\'target\\\'))\\n        h, w = a.shape[:2]\\n        top, left = ((h - REGION) // 2, (w - REGION) // 2)\\n        a = a[top:top + REGION, left:left + REGION].copy()\\n        gt = g[top:top + REGION, left:left + REGION].astype(np.float32) / 255\\n        x = a.astype(np.float32) / 255\\n        images = [(\\\'Input\\\', x)]\\n        folder = out / f\\\'{index:02d}\\\'\\n        folder.mkdir()\\n        Image.fromarray(a).save(folder / \\\'input.png\\\')\\n        Image.fromarray(g[top:top + REGION, left:left + REGION]).save(folder / \\\'gt.png\\\')\\n        base = {\\\'sample_id\\\': row[\\\'sample_id\\\'], \\\'index\\\': index, \\\'top\\\': top, \\\'left\\\': left, \\\'region\\\': REGION}\\n        results.append({**base, \\\'method\\\': \\\'input\\\', **metrics(x, gt), \\\'seconds\\\': 0.0})\\n        for scale in SCALES:\\n            torch.cuda.synchronize()\\n            start = time.perf_counter()\\n            try:\\n                pred = predict(model, a, scale)\\n            except torch.cuda.OutOfMemoryError:\\n                torch.cuda.empty_cache()\\n                raise RuntimeError(\\\'显存不足：当前实验停止，不混合不同区域的成绩。将 REGION 改成 512 后整轮重跑；结果目录会独立保存。\\\') from None\\n            torch.cuda.synchronize()\\n            seconds = time.perf_counter() - start\\n            method = f\\\'esd_scale_{scale:g}\\\'\\n            results.append({**base, \\\'method\\\': method, **metrics(pred, gt), \\\'seconds\\\': seconds})\\n            Image.fromarray(np.round(pred * 255).astype(\\\'uint8\\\')).save(folder / f\\\'{method}.png\\\')\\n            images.append((method, pred))\\n        preview(images + [(\\\'GT\\\', gt)], folder / \\\'comparison.jpg\\\')\\n        pd.DataFrame(results).to_csv(out / \\\'per_image.csv\\\', index=False)\\n    frame = pd.DataFrame(results)\\n    summary = frame.groupby(\\\'method\\\').agg(psnr=(\\\'psnr\\\', \\\'mean\\\'), ssim=(\\\'ssim\\\', \\\'mean\\\'), images=(\\\'index\\\', \\\'nunique\\\'))\\n    summary.to_csv(out / \\\'summary.csv\\\')\\n    pivot = frame.pivot(index=\\\'sample_id\\\', columns=\\\'method\\\', values=\\\'psnr\\\')\\n    delta = pivot.sub(pivot[\\\'input\\\'], axis=0).drop(columns=\\\'input\\\')\\n    delta.to_csv(out / \\\'gain_vs_input.csv\\\')\\n    display(summary)\\n    print(\\\'相对输入的逐图 PSNR 增益统计：\\\')\\n    display(delta.agg([\\\'mean\\\', \\\'median\\\', \\\'min\\\', \\\'max\\\']))\\n    print(\\\'正增益图片数：\\\')\\n    print((delta > 0).sum())\\n    with Image.open(out / \\\'00/comparison.jpg\\\') as im:\\n        display(im.copy())\\n    bundle = out.parent / (out.name + \\\'_results.zip\\\')\\n    with zipfile.ZipFile(bundle, \\\'w\\\', zipfile.ZIP_DEFLATED) as z:\\n        for p in out.rglob(\\\'*\\\'):\\n            if p.is_file():\\n                z.write(p, p.relative_to(out))\\n    print(\\\'完成。当前仅为中心区域开发诊断；不训练、不使用正式测试集、不宣称实际加速。\\\')\\n    print(\\\'下载并上传此 ZIP：\\\', bundle)\\n    display(FileLink(str(bundle)))\\n    return out\', H.__dict__)\nfrom pathlib import Path\nimport io, csv, json, random, hashlib, zipfile, time, copy\nfrom datetime import datetime\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom tqdm.auto import tqdm\nfrom IPython.display import display, FileLink\nDCID_SOURCE = None\nFUSION_ZIP = None\nWEIGHT = None\nUHDM_ROOT = None\nSTEPS = 300\nEVAL_EVERY = 100\nDCID_TRAIN = 12\nSOURCE_REPLAY = 20\nREGION = 768\nCROP = 256\nBATCH = 2\nLR = 1e-05\nSEED = 2026\nRUN_ROOT = Path(\'/kaggle/working/joint_moire_runs\')\nexec("def fusion_lowpass(x):\\n    v = torch.arange(-4, 5, device=x.device, dtype=x.dtype)\\n    k = torch.exp(-v.square() / 8)\\n    k = k / k.sum()\\n    c = x.shape[1]\\n    a = F.conv2d(F.pad(x, (4, 4, 0, 0), mode=\'replicate\'), k.view(1, 1, 1, 9).repeat(c, 1, 1, 1), groups=c)\\n    return F.conv2d(F.pad(a, (0, 0, 4, 4), mode=\'replicate\'), k.view(1, 1, 9, 1).repeat(c, 1, 1, 1), groups=c)\\n\\nclass ScaleFusion(nn.Module):\\n\\n    def __init__(self, mode=\'band\', preferred=0):\\n        super().__init__()\\n        assert mode in (\'spatial\', \'band\')\\n        self.mode = mode\\n        outputs = 6 if mode == \'band\' else 3\\n        self.net = nn.Sequential(nn.Conv2d(12, 24, 3, padding=1), nn.ReLU(), nn.Conv2d(24, 24, 3, padding=1), nn.ReLU(), nn.Conv2d(24, outputs, 1))\\n        nn.init.zeros_(self.net[-1].weight)\\n        nn.init.zeros_(self.net[-1].bias)\\n        with torch.no_grad():\\n            self.net[-1].bias[preferred] = 6.0\\n            if mode == \'band\':\\n                self.net[-1].bias[preferred + 3] = 6.0\\n\\n    def forward(self, x, candidates, lows):\\n        b, _, _, h, w = candidates.shape\\n        features = torch.cat((x, candidates.reshape(b, 9, h, w)), dim=1)\\n        features = F.interpolate(features, size=(max(1, h // 4), max(1, w // 4)), mode=\'area\')\\n        logits = F.interpolate(self.net(features), size=(h, w), mode=\'bilinear\', align_corners=False)\\n        wl = logits[:, :3].softmax(dim=1).unsqueeze(2)\\n        wh = logits[:, 3:].softmax(dim=1).unsqueeze(2) if self.mode == \'band\' else wl\\n        return (wl * lows + wh * (candidates - lows)).sum(dim=1)", globals())\n\ndef find_unique(pattern, explicit):\n    if explicit is not None:\n        p = Path(explicit)\n        if not p.is_file():\n            raise FileNotFoundError(p)\n        return p\n    found = sorted((p for root in [Path(\'/kaggle/input\'), Path(\'/kaggle/working\')] if root.exists() for p in root.rglob(pattern)))\n    if len(found) != 1:\n        raise RuntimeError(f\'请在配置中填写 {pattern} 的路径，候选={found}\')\n    return found[0]\n\ndef load_old_fusion():\n    if FUSION_ZIP is not None and Path(FUSION_ZIP).is_dir():\n        folder = Path(FUSION_ZIP)\n    else:\n        found = [Path(FUSION_ZIP)] if FUSION_ZIP is not None else [p for root in [Path(\'/kaggle/input\'), Path(\'/kaggle/working\')] if root.exists() for p in root.rglob(\'fusion_training_results.zip\')]\n        if len(found) > 1:\n            raise RuntimeError(f\'旧融合包不唯一，请填写 FUSION_ZIP：{found}\')\n        if found:\n            with zipfile.ZipFile(found[0]) as z:\n                p = z.read(\'protocol.json\')\n                ck = z.read(\'band/best.pt\')\n            return (json.loads(p), torch.load(io.BytesIO(ck), map_location=\'cpu\', weights_only=True), hashlib.sha256(p + ck).hexdigest())\n        folders = sorted({p.parent.parent for root in [Path(\'/kaggle/input\'), Path(\'/kaggle/working\')] if root.exists() for p in root.rglob(\'best.pt\') if p.parent.name == \'band\' and (p.parent.parent / \'protocol.json\').is_file()})\n        if len(folders) != 1:\n            raise RuntimeError(f\'请挂载旧融合结果包，FUSION_ZIP 可填写 ZIP 路径或展开后的根目录。候选={folders}\')\n        folder = folders[0]\n    p = (folder / \'protocol.json\').read_bytes()\n    ck = (folder / \'band/best.pt\').read_bytes()\n    return (json.loads(p), torch.load(io.BytesIO(ck), map_location=\'cpu\', weights_only=True), hashlib.sha256(p + ck).hexdigest())\n\ndef source_path(old, root):\n    p = Path(old)\n    if p.is_file():\n        return p\n    tail = old.replace(\'\\\\\', \'/\').split(\'/uhdm-dataset/\', 1)[-1]\n    direct = root / tail\n    if direct.is_file():\n        return direct\n    matches = [p for p in root.rglob(Path(tail).name) if p.parent.name == Path(tail).parent.name]\n    if len(matches) != 1:\n        raise RuntimeError(f\'UHDM 图片无法唯一映射：{old}; {matches}\')\n    return matches[0]\n\ndef crop_pair(a, g, label):\n    if a.shape != g.shape or min(a.shape[:2]) < REGION:\n        raise RuntimeError(f\'尺寸异常：{label}\')\n    h, w = a.shape[:2]\n    t, l = ((h - REGION) // 2, (w - REGION) // 2)\n    return (a[t:t + REGION, l:l + REGION].copy(), g[t:t + REGION, l:l + REGION].copy())\n\ndef load_source(items, root, split):\n    output = []\n    for row in tqdm(items, desc=\'UHDM \' + split):\n        xp, gp = (source_path(row[\'x\'], root), source_path(row[\'gt\'], root))\n        with Image.open(xp) as im:\n            a = np.array(im.convert(\'RGB\'))\n        with Image.open(gp) as im:\n            g = np.array(im.convert(\'RGB\'))\n        a, g = crop_pair(a, g, str(gp))\n        output.append({\'id\': \'UHDM/\' + gp.parent.name + \'/\' + gp.stem, \'x\': a, \'gt\': g, \'gt_hash\': H.sha_file(gp), \'input_hash\': H.sha_file(xp), \'path\': str(gp), \'split\': split, \'group\': str(row[\'group\'])})\n    return output\n\ndef tensor(a):\n    return torch.from_numpy(a.copy()).permute(2, 0, 1)[None].float().cuda() / 255\n\ndef anchor(y, x):\n    return y + (x - y).mean(dim=(-2, -1), keepdim=True)\n\n@torch.inference_mode()\ndef forward_native(model, a):\n    x = tensor(a)\n    h, w = x.shape[-2:]\n    y = model(F.pad(x, (0, -w % 32, 0, -h % 32), mode=\'replicate\'))[0][:, :, :h, :w]\n    return (y, x)\n\ndef arr(t):\n    return t[0].detach().clamp(0, 1).permute(1, 2, 0).cpu().numpy()\n\n@torch.inference_mode()\ndef evaluate(model, items, label, domain, out, step, anchored=False, include_baselines=False, fusion=None):\n    model.eval()\n    records = []\n    for i, p in enumerate(items):\n        raw, x = forward_native(model, p[\'x\'])\n        g = p[\'gt\'].astype(np.float32) / 255\n        predictions = {label: arr(anchor(raw, x) if anchored else raw)}\n        if include_baselines:\n            predictions = {\'input\': p[\'x\'].astype(np.float32) / 255, \'original\': arr(raw), \'mean_anchor\': arr(anchor(raw.clamp(0, 1), x))}\n            if fusion is not None:\n                candidates = torch.stack([torch.from_numpy(H.predict(model, p[\'x\'], s)).permute(2, 0, 1).cuda() for s in (0.5, 0.75, 1.0)])[None]\n                lows = fusion_lowpass(candidates[0])[None]\n                predictions[\'old_band\'] = arr(fusion(x, candidates, lows))\n        for method, pred in predictions.items():\n            score = H.metrics(pred, g)\n            records.append({\'domain\': domain, \'id\': p[\'id\'], \'method\': method, \'step\': step, **score})\n            dest = out / \'images\' / domain / method\n            dest.mkdir(parents=True, exist_ok=True)\n            Image.fromarray(np.round(pred * 255).astype(\'uint8\')).save(dest / f\'{i:02d}.png\')\n        if i < 3:\n            dest = out / \'previews\'\n            dest.mkdir(exist_ok=True)\n            H.preview([(\'Input\', p[\'x\'] / 255)] + list(predictions.items()) + [(\'GT\', g)], dest / f\'{domain}_{label}_{step}_{i}.jpg\')\n    return records\n\ndef check_split(training, validation):\n    for field in [\'gt_hash\', \'input_hash\']:\n        overlap = {p[field] for p in training} & {p[field] for p in validation}\n        if overlap:\n            raise RuntimeError(f\'训练/验证有精确重复 {field}；请修复划分，尚未启动训练\')\n\ndef draw_batch(items, rng):\n    xs = []\n    gs = []\n    for _ in range(BATCH):\n        p = rng.choice(items)\n        t = rng.randrange(REGION - CROP + 1)\n        l = rng.randrange(REGION - CROP + 1)\n        a = p[\'x\'][t:t + CROP, l:l + CROP]\n        g = p[\'gt\'][t:t + CROP, l:l + CROP]\n        if rng.random() < 0.5:\n            a, g = (np.flip(a, 1), np.flip(g, 1))\n        xs.append(tensor(a))\n        gs.append(tensor(g))\n    return (torch.cat(xs), torch.cat(gs))\n\ndef save_bundle(out):\n    bundle = out.parent / (out.name + \'_results.zip\')\n    with zipfile.ZipFile(bundle, \'w\', zipfile.ZIP_DEFLATED) as z:\n        for p in out.rglob(\'*\'):\n            if p.is_file():\n                z.write(p, p.relative_to(out))\n    print(\'结果包：\', bundle)\n    display(FileLink(str(bundle)))\n\ndef run_joint():\n    assert torch.cuda.is_available(), \'需要 GPU；CPU 会话先保存数据即可\'\n    assert REGION % 32 == 0 and CROP % 32 == 0 and (CROP <= REGION)\n    protocol, checkpoint, old_fusion_hash = load_old_fusion()\n    if UHDM_ROOT is None:\n        roots = sorted((p for p in Path(\'/kaggle/input\').rglob(\'uhdm-dataset\') if p.is_dir()))\n        if len(roots) != 1:\n            raise RuntimeError(\'请 Add Input 挂载 soumikrakshit/uhdm-dataset，或填写 UHDM_ROOT\')\n        source_root = roots[0]\n    else:\n        source_root = Path(UHDM_ROOT)\n    H.DATA_SOURCE = DCID_SOURCE\n    source = H.PairSource(H.locate_source())\n    rows = sorted(source.rows, key=lambda r: r[\'sample_id\'])\n    random.Random(SEED).shuffle(rows)\n    assert len(rows) == 20, \'首轮固定使用已检查的 20 对，不自动加入其他图片\'\n    target = []\n    for i, row in enumerate(tqdm(rows, desc=\'DCID 配对\')):\n        a, g = crop_pair(source.read(row, \'input\'), source.read(row, \'target\'), row[\'sample_id\'])\n        target.append({\'id\': \'DCID/\' + row[\'sample_id\'], \'x\': a, \'gt\': g, \'gt_hash\': hashlib.sha256(source.read_bytes(row, \'target\')).hexdigest(), \'input_hash\': hashlib.sha256(source.read_bytes(row, \'input\')).hexdigest(), \'split\': \'train\' if i < DCID_TRAIN else \'development_val\', \'path\': row[\'target\']})\n    target_train, target_val = (target[:DCID_TRAIN], target[DCID_TRAIN:])\n    source_train = load_source(protocol[\'manifest\'][\'train\'][:SOURCE_REPLAY], source_root, \'replay_train\')\n    source_val = load_source(protocol[\'manifest\'][\'val\'], source_root, \'old_development_val\')\n    check_split(source_train + target_train, source_val + target_val)\n    if {p[\'group\'] for p in source_train} & {p[\'group\'] for p in source_val}:\n        raise RuntimeError(\'UHDM 旧组划分有交集\')\n    out = RUN_ROOT / datetime.now().strftime(\'joint_%Y%m%d_%H%M%S\')\n    out.mkdir(parents=True)\n    H.ROOT = RUN_ROOT / \'official\'\n    H.ROOT.mkdir(exist_ok=True)\n    H.LOCAL_WEIGHT = WEIGHT\n    model, metadata = H.load_official_model()\n    digest = hashlib.sha256()\n    for k, v in sorted(model.state_dict().items()):\n        digest.update(k.encode())\n        digest.update(v.detach().cpu().contiguous().numpy().tobytes())\n    if digest.hexdigest() != protocol[\'weight_sha256\']:\n        raise RuntimeError(\'原始主干与旧融合实验权重不一致，停止，不能混接\')\n    band = ScaleFusion(\'band\', int(checkpoint[\'settings\'][\'preferred\'])).cuda().eval()\n    band.load_state_dict(checkpoint[\'model\'], strict=True)\n    for p in band.parameters():\n        p.requires_grad_(False)\n    manifest = [{k: v for k, v in p.items() if k not in [\'x\', \'gt\']} for p in target + source_train + source_val]\n    pd.DataFrame(manifest).to_csv(out / \'manifest.csv\', index=False)\n    settings = {**metadata, \'seed\': SEED, \'region\': REGION, \'crop\': CROP, \'batch\': BATCH, \'steps_per_branch\': STEPS, \'lr\': LR, \'source_replay_images\': len(source_train), \'source_val_images\': len(source_val), \'target_train_images\': len(target_train), \'target_val_images\': len(target_val), \'replay_schedule\': \'three DCID batches then one UHDM batch; total optimizer steps equal\', \'trainable\': \'decoder only; encoder frozen eval; MSE on unclipped input-mean-anchored prediction; adaptation eval anchors raw output then clips; fixed mean_anchor baseline anchors clipped original prediction\', \'old_fusion_protocol_and_band_sha256\': old_fusion_hash, \'old_protocol\': protocol, \'torch\': torch.__version__, \'gpu\': torch.cuda.get_device_name(0), \'limitations\': \'DCID image split, scene grouping unknown, prior inspected data; UHDM old development val; no independent test claim. Fixed last step is primary; best target checkpoint is exploratory. Same total steps means different target update counts.\'}\n    (out / \'protocol.json\').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    initial = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n    records = []\n    history = []\n    try:\n        print(\'阶段 1：两数据集固定基线与旧融合，先得到可下载结果\')\n        for domain, items in [(\'DCID\', target_val), (\'UHDM\', source_val)]:\n            records += evaluate(model, items, \'baseline\', domain, out, 0, include_baselines=True, fusion=band)\n        del band\n        torch.cuda.empty_cache()\n\n        def persist():\n            frame = pd.DataFrame(records)\n            frame.to_csv(out / \'per_image.csv\', index=False)\n            table = frame.groupby([\'domain\', \'method\', \'step\']).agg(psnr=(\'psnr\', \'mean\'), ssim=(\'ssim\', \'mean\'), images=(\'id\', \'nunique\'))\n            table.to_csv(out / \'summary.csv\')\n            pd.DataFrame(history).to_csv(out / \'history.csv\', index=False)\n            return table\n        display(persist())\n        save_bundle(out)\n        for branch in [\'target_only\', \'target_with_replay\']:\n            model.load_state_dict(initial, strict=True)\n            model.requires_grad_(False)\n            model.decoder.requires_grad_(True)\n            opt = torch.optim.Adam(model.decoder.parameters(), lr=LR)\n            rng = random.Random(SEED)\n            torch.manual_seed(SEED)\n            torch.cuda.manual_seed_all(SEED)\n            best = -float(\'inf\')\n            begin = time.perf_counter()\n            for step in tqdm(range(1, STEPS + 1), desc=branch):\n                model.train()\n                model.encoder.eval()\n                pool = source_train if branch == \'target_with_replay\' and step % 4 == 0 else target_train\n                x, g = draw_batch(pool, rng)\n                opt.zero_grad(set_to_none=True)\n                y = model(x)[0]\n                loss = F.mse_loss(anchor(y, x), g)\n                if not torch.isfinite(loss):\n                    raise RuntimeError(\'loss 非有限值，停止\')\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.decoder.parameters(), 1.0)\n                opt.step()\n                if step == 20:\n                    torch.cuda.synchronize()\n                    elapsed = time.perf_counter() - begin\n                    print(f\'{branch} 前20步 {elapsed:.1f}s；仅训练部分预计剩余 {(STEPS - step) * elapsed / 20 / 60:.1f} 分钟，另有评价开销\')\n                if step % EVAL_EVERY == 0 or step == STEPS:\n                    metrics = []\n                    for domain, items in [(\'DCID\', target_val), (\'UHDM\', source_val)]:\n                        metrics += evaluate(model, items, branch, domain, out, step, anchored=True)\n                    records += metrics\n                    score = float(np.mean([r[\'psnr\'] for r in metrics if r[\'domain\'] == \'DCID\']))\n                    history.append({\'branch\': branch, \'step\': step, \'train_loss\': float(loss.detach()), \'target_val_psnr\': score, \'elapsed_seconds\': time.perf_counter() - begin})\n                    payload = {\'decoder\': model.decoder.state_dict(), \'step\': step, \'branch\': branch, \'settings\': {k: v for k, v in settings.items() if k != \'old_protocol\'}}\n                    torch.save(payload, out / (branch + \'_last.pt\'))\n                    if score > best:\n                        best = score\n                        torch.save(payload, out / (branch + \'_best_target_dev.pt\'))\n                    display(persist())\n            del opt\n            save_bundle(out)\n        print(\'完成：优先比较两个分支固定第300步与 mean_anchor；同时查看 UHDM 是否退化。\')\n        return out\n    except Exception as error:\n        (out / \'error.txt\').write_text(repr(error), encoding=\'utf-8\')\n        if records:\n            pd.DataFrame(records).to_csv(out / \'partial_per_image.csv\', index=False)\n        save_bundle(out)\n        raise',J.__dict__)
from pathlib import Path
import json,random,hashlib,time,zipfile
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm
from IPython.display import display,FileLink

# 与上一轮挂载相同的三个 Input。通常保持自动搜索即可。
DCID_SOURCE=None
FUSION_ZIP=None      # 旧融合ZIP，或展开后含 protocol.json、band 的目录
UHDM_ROOT=None
WEIGHT=None
REGION=768
SEED=2026
RIDGE_ALPHAS=(1.,10.,100.)
PRIMARY_ALPHA=10.   # 预先指定主结果；另外两档是开发敏感性分析，不自动挑最好
MAX_SHIFT=0.25      # 每通道全局校准幅度上限；对所有方法一致
ROOT=Path('/kaggle/working/color_calibration')

def fingerprint(model):
    digest=hashlib.sha256()
    for k,v in sorted(model.state_dict().items()):
        digest.update(k.encode());digest.update(v.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def prepare():
    assert torch.cuda.is_available(),'缓存预测需要GPU；缓存完成后的拟合与评价使用CPU'
    J.REGION=REGION;J.DCID_SOURCE=DCID_SOURCE;J.FUSION_ZIP=FUSION_ZIP
    protocol,ck,old_hash=J.load_old_fusion()
    if UHDM_ROOT is None:
        roots=sorted(p for p in Path('/kaggle/input').rglob('uhdm-dataset') if p.is_dir())
        if len(roots)!=1:raise RuntimeError('请挂载 soumikrakshit/uhdm-dataset，或设置 UHDM_ROOT')
        root=roots[0]
    else:root=Path(UHDM_ROOT)
    J.H.DATA_SOURCE=DCID_SOURCE;source=J.H.PairSource(J.H.locate_source())
    rows=sorted(source.rows,key=lambda r:r['sample_id']);random.Random(SEED).shuffle(rows)
    assert len(rows)==20,'本轮固定复用原20对，不能悄悄更换划分'
    target=[]
    for i,r in enumerate(tqdm(rows,desc='读取DCID')):
        a,g=J.crop_pair(source.read(r,'input'),source.read(r,'target'),r['sample_id'])
        target.append({'id':'DCID/'+r['sample_id'],'domain':'DCID','split':'train' if i<12 else 'val',
            'x':a,'gt':g,'gt_hash':hashlib.sha256(source.read_bytes(r,'target')).hexdigest(),
            'input_hash':hashlib.sha256(source.read_bytes(r,'input')).hexdigest()})
    train=J.load_source(protocol['manifest']['train'][:20],root,'train')
    val=J.load_source(protocol['manifest']['val'],root,'val')
    for p in train+val:p['domain']='UHDM'
    items=target+train+val
    J.check_split([p for p in items if p['split']=='train'],[p for p in items if p['split']=='val'])
    if {p['group'] for p in train}&{p['group'] for p in val}:raise RuntimeError('UHDM组冲突')
    J.H.ROOT=ROOT/'official';J.H.ROOT.mkdir(parents=True,exist_ok=True);J.H.LOCAL_WEIGHT=WEIGHT
    if WEIGHT is None:
        known=[Path('/kaggle/working/joint_moire_runs/official/uhdm_checkpoint.pth'),
               Path('/kaggle/working/dcid_baseline/uhdm_checkpoint.pth')]
        J.H.LOCAL_WEIGHT=next((p for p in known if p.is_file()),None)
    model,meta=J.H.load_official_model()
    assert fingerprint(model)==protocol['weight_sha256'],'主干权重与旧融合协议不匹配'
    band=J.ScaleFusion('band',int(ck['settings']['preferred'])).cuda().eval()
    band.load_state_dict(ck['model'],strict=True);band.requires_grad_(False)
    return items,model,band,{**meta,'old_fusion_hash':old_hash}

@torch.inference_mode()
def cache_predictions(items,model,band,meta):
    signature={'meta':meta,'region':REGION,'protocol':'clipped_float16_original_and_band_v1',
        'samples':[{k:p[k] for k in ['id','split','domain','input_hash','gt_hash']} for p in items]}
    key=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()[:16]
    folder=ROOT/'cache'/key;folder.mkdir(parents=True,exist_ok=True)
    (folder/'signature.json').write_text(json.dumps(signature,indent=2),encoding='utf-8')
    records=[]
    for i,p in enumerate(tqdm(items,desc='缓存两种固定输出（可复用）')):
        dest=folder/f'{i:03d}.npz'
        if not dest.is_file():
            x=J.tensor(p['x'])
            c=torch.stack([torch.from_numpy(J.H.predict(model,p['x'],s)).permute(2,0,1).cuda() for s in (.5,.75,1.)])[None]
            lows=J.fusion_lowpass(c[0])[None]
            old=J.arr(band(x,c,lows));original=J.arr(c[:,2])
            tmp=dest.with_suffix('.tmp')
            with tmp.open('wb') as f:np.savez(f,x=p['x'],gt=p['gt'],original=original.astype(np.float16),old_band=old.astype(np.float16))
            tmp.replace(dest)
        records.append({k:v for k,v in p.items() if k not in ['x','gt']}|{'cache':str(dest)})
    return records,signature

def features(x,b):
    # 18维，仅使用输入和基础预测；无GT、无domain标签。
    axes=(0,1)
    return np.concatenate([x.mean(axes),x.std(axes),b.mean(axes),b.std(axes),
                           (x-b).mean(axes),(x-b).std(axes)]).astype(np.float64)

def weighted_fit(X,Y,domains,alpha):
    # 每域总权重相同；所有统计和标签仅来自训练池。
    groups=sorted(set(domains));w=np.array([1./(len(groups)*sum(d==g for d in domains)) for g in domains])
    mean=(w[:,None]*X).sum(0);std=np.sqrt((w[:,None]*(X-mean)**2).sum(0));std=np.maximum(std,.01)
    A=np.column_stack([(X-mean)/std,np.ones(len(X))])
    penalty=np.eye(A.shape[1])*alpha;penalty[-1,-1]=0.
    # 平均加权误差 + alpha/N * ||coeff||²。截距不惩罚。
    # 用同一PyTorch CPU线性代数后端，避免Windows本地NumPy/PyTorch的MKL运行时冲突。
    at=torch.from_numpy(A);wt=torch.from_numpy(w[:,None]);yt=torch.from_numpy(Y)
    coeff=torch.linalg.solve(at.T@(wt*at)+torch.from_numpy(penalty)/len(X),at.T@(wt*yt)).numpy()
    return {'mean':mean,'std':std,'coeff':coeff}

def infer_shift(f,fit):
    row=np.r_[((f-fit['mean'])/fit['std']),1.]
    return np.clip((row[:,None]*fit['coeff']).sum(0),-MAX_SHIFT,MAX_SHIFT)

def evaluate(records,fits,domain_means,out):
    rows=[]
    for p in tqdm([r for r in records if r['split']=='val'],desc='CPU评价'):
        with np.load(p['cache']) as a:
            x=a['x'].astype(np.float32)/255;gt=a['gt'].astype(np.float32)/255
            predictions={'input':x}
            for base in ['original','old_band']:
                b=a[base].astype(np.float32);feat=features(x,b)
                predictions[base]=b
                predictions[base+'_zero_shift']=np.clip(b+np.zeros(3,dtype=np.float32),0,1)
                # 原样保留已验证的固定均值规则，不受学习校准器幅度上限限制。
                predictions[base+'_mean_anchor']=np.clip(b+(x-b).mean((0,1),keepdims=True),0,1)
                predictions[base+'_domain_constant']=np.clip(b+np.clip(domain_means[(base,p['domain'])],-MAX_SHIFT,MAX_SHIFT),0,1)
                for alpha in RIDGE_ALPHAS:
                    predictions[f'{base}_ridge_{alpha:g}']=np.clip(b+infer_shift(feat,fits[(base,alpha)]),0,1)
        for method,pred in predictions.items():
            rows.append({'domain':p['domain'],'id':p['id'],'method':method,**J.H.metrics(pred,gt)})
        i=sum(r['domain']==p['domain'] and r['split']=='val' and r['id']<p['id'] for r in records)
        # 所有验证图都保存核心对照；不依赖GT选择展示方法。
        chosen=['input','original','original_mean_anchor',f'original_ridge_{PRIMARY_ALPHA:g}',
                'old_band',f'old_band_ridge_{PRIMARY_ALPHA:g}']
        for method in chosen:
            dest=out/'images'/p['domain']/method;dest.mkdir(parents=True,exist_ok=True)
            Image.fromarray(np.round(predictions[method]*255).astype('uint8')).save(dest/f'{i:02d}.png')
        dest=out/'previews';dest.mkdir(exist_ok=True)
        J.H.preview([(k,predictions[k]) for k in chosen]+[('GT',gt)],dest/f'{p["domain"]}_{i:02d}.jpg')
        pd.DataFrame(rows).to_csv(out/'per_image.csv',index=False)
    frame=pd.DataFrame(rows)
    # 校准结构的真正零初始化与基础输出必须完全一致。
    for base in ['original','old_band']:
        a=frame[frame.method==base].set_index('id');b=frame[frame.method==base+'_zero_shift'].set_index('id')
        assert np.allclose(a.psnr,b.psnr,atol=1e-9),'零修正对照不一致'
    table=frame.groupby(['domain','method']).agg(psnr=('psnr','mean'),ssim=('ssim','mean'),images=('id','nunique'))
    table.to_csv(out/'summary.csv');display(table)
    return frame

def run_calibration():
    assert PRIMARY_ALPHA in RIDGE_ALPHAS
    ROOT.mkdir(parents=True,exist_ok=True)
    out=ROOT/datetime.now().strftime('calibration_%Y%m%d_%H%M%S');out.mkdir()
    try:
        items,model,band,meta=prepare()
        records,signature=cache_predictions(items,model,band,meta)
        model.cpu();band.cpu();del model,band,items;torch.cuda.empty_cache()
        print('固定预测缓存完成；后续在CPU拟合及评价，不再反向传播主干。')
        pd.DataFrame(records).to_csv(out/'manifest.csv',index=False)
        settings={'signature':signature,'primary_alpha':PRIMARY_ALPHA,'alphas':RIDGE_ALPHAS,
            'max_shift':MAX_SHIFT,'fit':'18 input/prediction statistics -> 3 global RGB shifts; ridge; equal domain weight; train only',
            'gt_label':'mean(GT - clipped_base) over native center768; preclipping global MSE optimum',
            'domain_constant':'separate train mean shift per known dataset; uses dataset label, only a comparator',
            'precision':'float16 frozen predictions cached then float32 evaluation; not pixel-identical to prior float32 baseline',
            'limits':'same 12/8 DCID image split and 20/10 UHDM development split as previous run; no independent test; scene overlap unknown; alpha10 primary, others development sensitivity; not guaranteed gain'}
        (out/'protocol.json').write_text(json.dumps(settings,ensure_ascii=False,indent=2),encoding='utf-8')
        train=[p for p in records if p['split']=='train'];fits={};domain_means={};feature_rows=[]
        for base in ['original','old_band']:
            X=[];Y=[];domains=[]
            for p in train:
                with np.load(p['cache']) as a:
                    x=a['x'].astype(np.float64)/255;g=a['gt'].astype(np.float64)/255;b=a[base].astype(np.float64)
                    f=features(x,b);y=(g-b).mean((0,1))
                X.append(f);Y.append(y);domains.append(p['domain'])
                feature_rows.append({'id':p['id'],'domain':p['domain'],'base':base,
                    **{f'f{i}':v for i,v in enumerate(f)},**{f'label{i}':v for i,v in enumerate(y)}})
            X,Y=np.array(X),np.array(Y)
            for d in sorted(set(domains)):domain_means[(base,d)]=Y[np.array(domains)==d].mean(0)
            for alpha in RIDGE_ALPHAS:
                fit=weighted_fit(X,Y,domains,alpha);fits[(base,alpha)]=fit
                folder=out/'models';folder.mkdir(exist_ok=True)
                np.savez(folder/f'{base}_ridge_{alpha:g}.npz',**fit,max_shift=MAX_SHIFT)
        pd.DataFrame(feature_rows).to_csv(out/'train_statistics.csv',index=False)
        (out/'domain_constant.json').write_text(json.dumps({f'{b}/{d}':v.tolist() for (b,d),v in domain_means.items()},indent=2))
        evaluate(records,fits,domain_means,out)
        print('完成。先比较固定alpha10与各自base、mean_anchor；同时检查另一域有无退化，不自动挑验证集最高分。')
    except Exception as error:
        (out/'error.txt').write_text(repr(error),encoding='utf-8');raise
    finally:
        bundle=out.parent/(out.name+'_results.zip')
        with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
            for p in out.rglob('*'):
                if p.is_file():z.write(p,p.relative_to(out))
        print('结果ZIP：',bundle);display(FileLink(str(bundle)))
    return out

if __name__=='__main__':
    CALIBRATION_RUN=run_calibration()
